# AI YouTube Shorts Generator — Colab Ready

**Langkah:**
1. `Runtime > Change runtime type > T4 GPU > Save`
2. Isi key & endpoint di cell 4
3. Paste YouTube URL di cell 7
4. `Runtime > Run all`

In [ ]:
# 1. Pull + copy patched files
!git clone --depth 1 https://github.com/shycecil077/AI-Youtube-Shorts-Generator.git /content/shorts-gen 2>/dev/null || true
%cd /content/shorts-gen
!cp /workspace/shorts_generator/config.py /content/shorts-gen/shorts_generator/config.py
!cp /workspace/shorts_generator/local/llm.py /content/shorts-gen/shorts_generator/local/llm.py
!cp /workspace/shorts_generator/local/downloader.py /content/shorts-gen/shorts_generator/local/downloader.py
!cp /workspace/shorts_generator/highlights.py /content/shorts-gen/shorts_generator/highlights.py
print("[patch] All source files synced from workspace")


In [ ]:
# 2. Install dependencies
!pip install -q requests python-dotenv
!apt-get install -y ffmpeg > /dev/null 2>&1
!pip install -q yt-dlp 'yt-dlp[default]' faster-whisper openai google-genai opencv-python-headless
!pip install -q browser-cookie3 > /dev/null 2>&1 || true

In [ ]:
# 3. Verifikasi instalasi
import subprocess, sys, os, torch

for cmd in ["python", "ffmpeg", "yt-dlp"]:
    r = subprocess.run([cmd, "--version"], capture_output=True, text=True)
    print(f"{cmd}: {(r.stdout+r.stderr).split(chr(10))[0] or 'ok'}")

import requests, yt_dlp, faster_whisper, openai, cv2
print("All packages OK")
print(f"CUDA: {'YES' if torch.cuda.is_available() else 'NO'}")

In [ ]:
# 4. Extract YouTube cookies dari browser (jika tersedia)
import os as _os
_COOKIES_FILE = "/content/cookies.txt"

def _try_extract_cookies():
    try:
        import browser_cookie3
        cj = browser_cookie3.chrome(domain_name='.youtube.com')
        if cj:
            with open(_COOKIES_FILE, 'w') as f:
                for cookie in cj:
                    f.write(f"{cookie.domain}\t{'TRUE' if cookie.secure else 'FALSE'}\t{cookie.path}\t"
                            f"{'TRUE' if cookie.secure else 'FALSE'}\t{int(cookie.expires or 0)}\t{cookie.name}\t{cookie.value}\n")
            print(f"[cookies] extracted from Chrome: {len(cj)} cookies")
            return _COOKIES_FILE
    except Exception as e:
        print(f"[cookies] browser_cookie3 failed: {e}")
    return None

_cookie_path = _try_extract_cookies()
if not _cookie_path:
    print("[cookies] no browser cookies found (Colab headless — will try without cookies)")
os.environ["YOUTUBE_COOKIES"] = _cookie_path or ""

In [ ]:
# 5. Isi konfigurasi kamu di sini
os.environ["MUAPI_API_KEY"] = ""           # YOUR MUAPI KEY HERE
os.environ["MUAPI_BASE_URL"] = ""          # YOUR MUAPI ENDPOINT HERE
os.environ["LLM_PROVIDER"] = "custom"      # openai / gemini / custom
os.environ["OPENAI_API_KEY"] = ""           # OpenAI key (jika LLM_PROVIDER=openai)
os.environ["OPENAI_MODEL"] = "gpt-4o-mini"
os.environ["GEMINI_API_KEY"] = ""           # Gemini key (jika LLM_PROVIDER=gemini)
os.environ["GEMINI_MODEL"] = "gemini-2.5-flash"

# --- CUSTOM ENDPOINT (jika LLM_PROVIDER=custom) ---
os.environ["CUSTOM_API_KEY"] = ""           # API KEY KAMU SENDIRI
os.environ["CUSTOM_LLM_BASE_URL"] = ""      # BASE URL ENDPOINT KAMU
os.environ["CUSTOM_MODEL"] = ""             # NAMA MODEL DI ENDPOINT KAMU (opsional)

os.environ["LOCAL_WHISPER_MODEL"] = "large-v3"
os.environ["LOCAL_WHISPER_DEVICE"] = "auto"
os.environ["LOCAL_OUTPUT_DIR"] = "/content/shorts_output"
os.environ["SHOW_LLM_OUTPUT"] = "true"  # enable to see raw LLM responses

print("[config] Keys & endpoint set")

In [ ]:
# 5.5. Cek status cache + deteksi URL otomatis
import os as _os
_OUT_DIR = "/content/shorts_output"
print(f"[cache] Folder: {_OUT_DIR}")
if _os.path.exists(_OUT_DIR):
    files = _os.listdir(_OUT_DIR)
    videos = [f for f in files if f.startswith('source_') and f.endswith(('.mp4','.mkv','.webm'))]
    srts   = [f for f in files if f.startswith('source_') and f.endswith('.srt')]
    jsons  = [f for f in files if f.endswith('.json')]
    shorts = [f for f in files if f.startswith('short_') and f.endswith('.mp4')]
    print(f"[cache] Video sumber : {videos if videos else '(kosong)'}")
    print(f"[cache] Transcript  : {srts if srts else '(kosong)'}")
    print(f"[cache] Shorts jadi : {shorts if shorts else '(kosong)'}")
    print(f"[cache] JSON result : {jsons if jsons else '(kosong)'}")
    for f in videos + srts:
        fp = _os.path.join(_OUT_DIR, f)
        mb = _os.path.getsize(fp) / 1024 / 1024
        print(f"          {f}  ({mb:.1f} MB)")
    if videos:
        vid_id = videos[0].replace('source_','').rsplit('.',1)[0]
        auto_url = f"https://www.youtube.com/watch?v={vid_id}"
        print(f"\n[cache] Auto-detect URL: {auto_url}")
else:
    print("[cache] Folder belum ada")

print("\nJika video & SRT sudah ada, pipeline akan SKIP download & transkrip.")
print("Hanya jalankan highlight + crop saja.")


---
**Backup & Restore Progress**

File backup/restore script ada di:
- `/workspace/backup_script.txt` — copy paste ke cell baru, jalankan, hapus cell
- `/workspace/restore_script.txt` — copy paste ke cell baru, jalankan, hapus cell

Cara pakai:
1. Setelah pipeline berhasil (atau sebelum re-run), jalankan **backup_script** untuk simpan progress ke Google Drive
2. Jika ingin restore, jalankan **restore_script** — akan kembalikan video + transkrip ke `shorts_output/`
3. Setelah itu re-run cell pipeline (cell 8)

In [ ]:
# 6. Discover endpoints + decide mode
muapi_key     = os.environ.get("MUAPI_API_KEY", "").strip()
muapi_base    = os.environ.get("MUAPI_BASE_URL", "").rstrip("/")
openai_key    = os.environ.get("OPENAI_API_KEY", "").strip()
gemini_key    = os.environ.get("GEMINI_API_KEY", "").strip()
custom_key    = os.environ.get("CUSTOM_API_KEY", "").strip()
custom_base   = os.environ.get("CUSTOM_LLM_BASE_URL", "").strip()
custom_model  = os.environ.get("CUSTOM_MODEL", "").strip()
cookie_path   = os.environ.get("YOUTUBE_COOKIES", "")

headers = {"x-api-key": muapi_key, "Content-Type": "application/json"}

print("[discover] Testing your API endpoints...")
for path in ["/models", "/", "/v1/models"]:
    try:
        resp = requests.get(muapi_base + path, headers=headers, timeout=8)
        print(f"  {path:20s} -> {resp.status_code}")
        if resp.status_code == 200:
            try:
                data = resp.json()
                if isinstance(data, dict):
                    print(f"        keys: {list(data.keys())}")
            except Exception:
                print(f"        text: {resp.text[:150]}")
    except Exception as e:
        print(f"  {path:20s} -> ERROR: {str(e)[:60]}")

print("\n[pipeline] Testing required endpoints...")
for ep in ["youtube-download", "openai-whisper", "autocrop"]:
    try:
        resp = requests.post(f"{muapi_base}/{ep}", headers=headers, json={}, timeout=5)
        print(f"  {ep:20s} -> {resp.status_code}")
    except Exception as e:
        print(f"  {ep:20s} -> ERROR: {str(e)[:60]}")

# Decide mode
if muapi_key and muapi_base:
    try:
        resp = requests.post(f"{muapi_base}/youtube-download", headers=headers, json={}, timeout=5)
        if resp.status_code in (200, 201, 202):
            MODE = "api"
        else:
            raise Exception(f"status {resp.status_code}")
    except Exception:
        MODE = "local"
else:
    MODE = "local"

print(f"\n[mode] {MODE}")
if MODE == "local":
    if not (openai_key or gemini_key or (custom_key and custom_base)):
        print("[WARNING] Local mode needs at least one of:")
        print("  - OPENAI_API_KEY")
        print("  - GEMINI_API_KEY")
        print("  - CUSTOM_API_KEY + CUSTOM_LLM_BASE_URL")
    if cookie_path:
        print(f"[cookies] using: {cookie_path}")
    else:
        print("[cookies] none available — will try download without cookies first")

os.environ["LLM_PROVIDER"] = os.environ.get("LLM_PROVIDER", "openai")
device = "cuda" if torch.cuda.is_available() else "cpu"
os.environ["LOCAL_WHISPER_DEVICE"] = device
print(f"[whisper] device={device}")

In [ ]:
# 7. Input YouTube URL (auto-filled from cache if available)
import os as _os
_OUT_DIR = "/content/shorts_output"
# Auto-detect video file from cache
_cached_videos = []
_cached_url = ""
if _os.path.exists(_OUT_DIR):
    for f in _os.listdir(_OUT_DIR):
        if f.startswith("source_") and f.endswith(('.mp4','.mkv','.webm')):
            vid_id = f.replace('source_','').rsplit('.',1)[0]
            _cached_videos.append(f)
            _cached_url = f"https://www.youtube.com/watch?v={vid_id}"
# User can override by pasting new URL below
YOUTUBE_URL = _cached_url  # auto-filled from cache
NUM_CLIPS = 3
ASPECT_RATIO = "9:16"
FORMAT = "720"
# Uncomment & fill below to override with new URL
# YOUTUBE_URL = "https://www.youtube.com/watch?v=NEW_VIDEO_ID"
print(f"URL:    {YOUTUBE_URL or '(kosong - akan pakai cache)'}")
print(f"Mode:   {MODE}")
print(f"API:    {muapi_base or '(not set)'}")
print(f"LLM:    custom={bool(custom_key and custom_base)} openai={bool(openai_key)} gemini={bool(gemini_key)}")
print(f"Cache:  {', '.join(_cached_videos) if _cached_videos else '(kosong)'}")


In [ ]:
# 8. Run pipeline (hard reload + skip download/transcribe if cache exists)
import sys, json, importlib
from pathlib import Path
sys.path.insert(0, "/content/shorts-gen")
# HARD RELOAD: clear cached modules and re-import
for mod_name in list(sys.modules.keys()):
    if 'shorts_generator' in mod_name:
        del sys.modules[mod_name]
print('[reload] Cleared shorts_generator cache')
# VERIFY: check that debug code is actually in the loaded files
_verify_highlights = Path('/content/shorts-gen/shorts_generator/highlights.py').read_text()
_verify_llm = Path('/content/shorts-gen/shorts_generator/local/llm.py').read_text()
if 'SHOW_LLM_OUTPUT' in _verify_highlights:
    print('[verify] highlights.py has SHOW_LLM_OUTPUT debug code')
else:
    print('[verify] WARNING: highlights.py does NOT have debug code!')
if 'SHOW_LLM_OUTPUT' in _verify_llm:
    print('[verify] llm.py has SHOW_LLM_OUTPUT debug code')
else:
    print('[verify] WARNING: llm.py does NOT have debug code!')
from shorts_generator import generate_shorts
from shorts_generator.muapi import MuAPIError
_openai_key = os.environ.get("OPENAI_API_KEY", "").strip()
_gemini_key = os.environ.get("GEMINI_API_KEY", "").strip()
_custom_key = os.environ.get("CUSTOM_API_KEY", "").strip()
_custom_base = os.environ.get("CUSTOM_LLM_BASE_URL", "").strip()
# Detect cached files
_out_dir = "/content/shorts_output"
_cached_video = None
_cached_srt = None
if _out_dir and os.path.exists(_out_dir):
    for _f in os.listdir(_out_dir):
        if _f.startswith('source_') and _f.endswith('.mp4'):
            _cached_video = os.path.join(_out_dir, _f)
        elif _f.startswith('source_') and _f.endswith('.srt'):
            _cached_srt = os.path.join(_out_dir, _f)
print(f"[pipeline] running in {MODE} mode...")
if _cached_video:
    print(f"[cache] Found cached video: {_cached_video}")
if _cached_srt:
    print(f"[cache] Found cached transcript: {_cached_srt}")
_use_cache = False
if _cached_video and (not YOUTUBE_URL or YOUTUBE_URL == _cached_url):
    _use_cache = True
    print(f"[pipeline] Using cached files — skipping download & transcribe")
result = None
try:
    if _use_cache:
        from shorts_generator.highlights import get_highlights
        from shorts_generator.local.transcriber import _load_srt_cache
        from shorts_generator.local.clipper import crop_highlights_local
        _srt_path = Path(_cached_srt) if isinstance(_cached_srt, str) else _cached_srt
        transcript = _load_srt_cache(_srt_path) if _cached_srt else {'duration': 0, 'segments': []}
        if not transcript['segments']:
            print('[pipeline] Cache transcript empty, need to re-transcribe')
            _use_cache = False
        else:
            print(f"[cache] Loaded {len(transcript['segments'])} segments, {transcript['duration']:.0f}s")
            llm_fn = None
            if MODE == 'local':
                from shorts_generator.local.llm import call_local_llm
                llm_fn = call_local_llm
            # Call highlights with truncation-safe parsing
        try:
            highlights_result = get_highlights(transcript, num_clips=NUM_CLIPS, llm_fn=llm_fn)
        except RuntimeError as he:
            print(f"[highlights] Retrying with simpler prompt due to: {he}", flush=True)
            # Retry with reduced complexity
            from shorts_generator.highlights import call_highlight_api, detect_content_type
            content_info = detect_content_type(transcript, llm_fn=llm_fn)
            transcript_text = ""
            for seg in transcript.get('segments', [])[:50]:  # Use only first 50 segments
                transcript_text += f"[{seg['start']:.1f}s] {seg['text'].strip()}\n"
            highlights_result = call_highlight_api(
                transcript_text, content_info, transcript['duration'],
                num_clips=NUM_CLIPS, is_chunk=False, llm_fn=llm_fn
            )
            all_highlights = highlights_result.get('highlights', [])
            if not all_highlights:
                raise RuntimeError('Highlight generator returned zero clips')
            top = sorted(all_highlights, key=lambda h: int(h.get('score',0)), reverse=True)[:NUM_CLIPS]
            print(f"[pipeline] cropping {len(top)} of {len(all_highlights)} candidates")
            shorts = crop_highlights_local(_cached_video, top, aspect_ratio=ASPECT_RATIO)
            result = {
                'mode': 'local',
                'source_video_url': _cached_video,
                'transcript': transcript,
                'highlights': all_highlights,
                'shorts': shorts,
            }
            print('[pipeline] done (cache mode)')
    else:
        result = generate_shorts(
            youtube_url=YOUTUBE_URL,
            num_clips=NUM_CLIPS,
            aspect_ratio=ASPECT_RATIO,
            download_format=FORMAT,
            language=None,
            mode=MODE,
        )
        print(f"[pipeline] done in {result.get('mode')} mode")
except MuAPIError as e:
    print(f"[pipeline] MuAPI error: {e}")
    if MODE == 'api' and (_openai_key or _gemini_key or (_custom_key and _custom_base)):
        print(f"[pipeline] AUTO-FALLBACK -> local mode")
        result = generate_shorts(
            youtube_url=YOUTUBE_URL,
            num_clips=NUM_CLIPS,
            aspect_ratio=ASPECT_RATIO,
            download_format=FORMAT,
            language=None,
            mode='local',
        )
    else:
        raise
except Exception as e:
    print(f"[pipeline] error: {e}")
    raise
print("\n" + "=" * 72)
print(f"Mode:          {result.get('mode')}")
print(f"Source video:  {result['source_video_url']}")
print(f"Highlights:    {len(result['highlights'])} candidates -> kept top {len(result['shorts'])}")
print("=" * 72)
for i, s in enumerate(result['shorts'], 1):
    print(f"\n#{i}  score={s.get('score')}  {s.get('start_time'):.1f}s -> {s.get('end_time'):.1f}s")
    print(f"     title:  {s.get('title')}")
    print(f"     hook:   {s.get('hook_sentence')}")
    print(f"     reason: {s.get('virality_reason')}")
    clip = s.get('clip_url') or s.get('error', '(failed)')
    print(f"     clip:   {clip}")
with open('/content/result.json', 'w') as f:
    json.dump(result, f, indent=2)
print(f"\nJSON: /content/result.json")


In [ ]:
# 9. Preview output
from IPython.display import display, HTML, Video

for i, s in enumerate(result["shorts"], 1):
    clip_path = s.get('clip_url')
    if not clip_path:
        print(f"#{i} FAILED: {s.get('error')}")
        continue
    if os.path.exists(clip_path):
        print(f"\n--- Short #{i}: {s.get('title')} ---")
        display(Video(clip_path, width=360, height=640))
    else:
        print(f"\n--- Short #{i}: {s.get('title')} ---")
        print(f"URL: {clip_path}")
        display(HTML(f'<a href="{clip_path}" target="_blank">Download Short #{i}</a>'))

In [ ]:
# 10. Save to Google Drive
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
    dest = "/content/drive/MyDrive/shorts_output"
    !mkdir -p "$dest"
    !cp -r /content/shorts_output/* "$dest/" 2>/dev/null || true
    !cp /content/result.json "$dest/" 2>/dev/null || true
    print(f"Saved to: {dest}/")
    !ls -lh "$dest/"
except Exception as e:
    print(f"Drive skipped: {e}")

In [ ]:
# 11. List output
!ls -lh /content/shorts_output/ 2>/dev/null || echo "No output yet"
!ls -lh /content/result.json 2>/dev/null || echo "No result.json"